In [11]:
import pandas as pd
from datetime import date 

In [ ]:
# Задача: Посчитать окупаемость маркетинга за последние 12 месяцев

# Тратим деньги
# Привлекаем пользователей 
# Пользователи что-то покупают

# Когорты от даты регистрации
# Посчитаем
# - выручку и LTV
# - рекламные расходы
# - окупаемость маркетинга

In [ ]:
# Алгоритм построения когортного отчета
# 0. Формулируем задачу 
# 1. Определяем когорту (Событие и временной промежуток) - даты регистрации по месяцам 
# 2. Выделяем целевые метрики - выручка, LTV, ROAS
# 3. Выбираем подходящий формат отчета - возростной


In [58]:
users = pd.read_csv('users.csv', index_col='user_id')
users['user_registration_dt'] = pd.to_datetime(users['user_registration_dt'])
users['reg_month'] = users['user_registration_dt'].dt.to_period('M')
users[['user_registration_dt', 'reg_month']]


,user_registration_dt,reg_month
user_id,,
1425,2020-06-23,2020-06
1429,2020-08-15,2020-08
1434,2020-03-07,2020-03
1436,2020-09-04,2020-09
1439,2020-06-29,2020-06
...,...,...
3111,2020-06-07,2020-06
3116,2020-06-21,2020-06
3119,2020-07-15,2020-07


In [68]:
cohort_size = users.groupby('reg_month').agg(n_users = ('user_email', 'count'))

In [64]:
orders = pd.read_csv('orders.csv', index_col='order_id')
orders['order_created_dt'] = pd.to_datetime(orders['order_created_dt'])
orders['payment_month'] = orders['order_created_dt'].dt.to_period('M')
orders = (
    orders
    .groupby('user_idi')
    .agg(first_payment_at = ('order_created_dt', 'min'))
    .merge(orders, how='inner', left_index=True, right_on='user_idi')
)
orders['payment_month'] = orders['order_created_dt'].dt.to_period('M')
orders['first_payment_month'] = orders['first_payment_at'].dt.to_period('M')

orders = (
    orders
    .merge(users[['user_registration_dt', 'reg_month']],
            how='inner', left_on='user_idi', right_on='user_id'))
orders

orders['cohort_age_days'] = (orders['order_created_dt'] - orders['user_registration_dt']).dt.days
orders['cohort_age_month'] = orders['cohort_age_days'] // 30
orders.head(4)

,first_payment_at,user_idi,order_amount,order_created_dt,payment_month,first_payment_month,user_registration_dt,reg_month,cohort_age_days,cohort_age_month
0,2021-01-08,1443,6440,2021-01-08,2021-01,2021-01,2020-01-22,2020-01,352,11
1,2020-10-09,1446,6440,2020-10-09,2020-10,2020-10,2020-02-14,2020-02,238,7
2,2021-06-04,1451,6440,2021-06-04,2021-06,2021-06,2020-07-01,2020-07,338,11
3,2021-11-23,1469,6440,2021-11-23,2021-11,2021-11,2021-01-03,2021-01,324,10


In [73]:
cohort_size.head(20)

,n_users
reg_month,
2020-01,33
2020-02,36
2020-03,43
2020-04,45
2020-05,41
2020-06,44
2020-07,59
2020-08,49
2020-09,56


In [65]:
( # Способ построения когортного отчета через группировку
    orders
    .groupby(['first_payment_month', 'cohort_age_month'])
    .agg(revenue = ('order_amount', 'sum'))
    .tail(15)  # tail - это head наоборот, показывает с конца
)

revenue
first_payment_month cohort_age_month         
2021-07             8                    6440
                    9                   19320
                    10                  12880
                    11                   6440
                    12                   6440
                    13                  19900
2021-08             8                    6440
                    9                    6440
                    11                  12880
2021-09             8                    6440
                    9                   19320
2021-10             9                    6440
                    11                  12880
2021-11             10                  19320
2021-12             11                  32780

In [41]:
cohorts = (
    orders
    .pivot_table(
        index='first_payment_month',
        columns='cohort_age_month',
        values='order_amount',
        aggfunc='sum'
    )
)
cohorts.tail(15)

cohort_age_month,0,1,2,4,5,7,8,9,11
first_payment_month,,,,,,,,,
2020-10,228300.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6440.0
2020-11,195520.0,NaN,26340.0,NaN,NaN,6440.0,26340.0,NaN,NaN
2020-12,248200.0,NaN,6440.0,NaN,6440.0,NaN,NaN,NaN,NaN
2021-01,248200.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02,117080.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-03,77860.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-04,169760.0,NaN,6440.0,NaN,NaN,NaN,NaN,NaN,NaN
2021-05,117660.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-06,90740.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
